In [ ]:
import os
import numpy as np
from scipy.stats import ttest_rel, wilcoxon
import json

In [8]:
def collect_latest_runs_for_subdirs(model_dir, subdirs):
    """
    model_dir: .../EVENT_DISTANCE_JUDGE/falcon-7b-instruct
    subdirs: e.g. ["neutral_1","neutral_2","neutral_3"]

    Returns:
      ua_list: [(subdir, ua_results.jsonl), ...]
      ru_list: [(subdir, ru_results.jsonl), ...]
    """
    def find_latest_run(neutral_dir, prefix):
        runs = [
            d for d in os.listdir(neutral_dir)
            if d.startswith(prefix) and os.path.isdir(os.path.join(neutral_dir, d))
        ]
        if not runs:
            return None
        runs.sort()
        latest = runs[-1]
        fname = "ua_results.jsonl" if prefix.startswith("ua") else "ru_results.jsonl"
        fpath = os.path.join(neutral_dir, latest, fname)
        return fpath if os.path.isfile(fpath) else None

    ua_files = []
    ru_files = []

    for sd in subdirs:
        sd_path = os.path.join(model_dir, sd)
        if not os.path.isdir(sd_path):
            continue

        ua_path = find_latest_run(sd_path, "ua_run_")
        ru_path = find_latest_run(sd_path, "ru_run_")

        print("Latest runs are:")
        print(f" Subdir: {sd}")
        print(f"  UA: {ua_path}")
        print(f"  RU: {ru_path}")

        if ua_path:
            ua_files.append((sd, ua_path))
        if ru_path:
            ru_files.append((sd, ru_path))

    return ua_files, ru_files


In [9]:
import os, json

def load_all_examples_with_prefix(files_with_prefix):
    """
    files_with_prefix: list of (prefix, jsonl_path)
    Returns {new_id: obj} where new_id = f"{prefix}_{old_id}"
    """
    all_examples = {}

    for prefix, path in files_with_prefix:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                old_id = obj.get("id")
                new_id = f"{prefix}_{old_id}"
                obj["id"] = new_id
                all_examples[new_id] = obj

    return all_examples


In [ ]:


def scores_from_examples(examples: dict, score_key: str) -> dict:
    """
    examples: {id: obj}
    returns: {id: score}
    """
    out = {}
    for ex_id, obj in examples.items():
        if score_key in obj and obj[score_key] is not None:
            out[ex_id] = float(obj[score_key])
    return out

def ua_vs_ru_significance_from_examples(
    ua_examples: dict,
    ru_examples: dict,
    ua_key="score_ua_mt_vs_ua_human",
    ru_key="score_ru_mt_vs_ru_human",
):
    ua_scores = scores_from_examples(ua_examples, ua_key)
    ru_scores = scores_from_examples(ru_examples, ru_key)

    common = sorted(set(ua_scores) & set(ru_scores))
    if len(common) == 0:
        raise ValueError("No overlapping ids between UA and RU examples dicts.")

    x = np.array([ua_scores[i] for i in common], dtype=float)
    y = np.array([ru_scores[i] for i in common], dtype=float)

    diff = x - y

    res = {
        "n_ua": len(ua_scores),
        "n_ru": len(ru_scores),
        "n_common": len(common),
        "mean_UA": float(x.mean()),
        "mean_RU": float(y.mean()),
        "mean_diff_UA_minus_RU": float(diff.mean()),
        "median_diff_UA_minus_RU": float(np.median(diff)),
    }

    # Paired t-test
    t = ttest_rel(x, y)
    res["paired_ttest"] = {"t": float(t.statistic), "p": float(t.pvalue)}

    # Wilcoxon (may error if all diffs are 0)
    try:
        w = wilcoxon(x, y, zero_method="wilcox")
        res["wilcoxon"] = {"W": float(w.statistic), "p": float(w.pvalue)}
    except ValueError as e:
        res["wilcoxon"] = {"error": str(e)}

    return res


In [21]:
import numpy as np
from scipy.stats import ttest_rel, wilcoxon, t as student_t

def bh_fdr(pvals):
    """
    Benjamini–Hochberg FDR correction.
    Returns q-values aligned with input order.
    """
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = ranked * m / (np.arange(1, m + 1))

    # enforce monotonicity
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0.0, 1.0)

    qvals = np.empty_like(q)
    qvals[order] = q
    return qvals


def paired_mean_ci(diff, alpha=0.05):
    """
    diff: 1D array of paired differences (e.g., RU-UA)
    Returns (mean, (lo, hi))
    """
    diff = np.asarray(diff, dtype=float)
    n = diff.size
    mean = diff.mean()
    if n < 2:
        return float(mean), (np.nan, np.nan)

    se = diff.std(ddof=1) / np.sqrt(n)
    tcrit = student_t.ppf(1 - alpha/2, df=n-1)
    lo = mean - tcrit * se
    hi = mean + tcrit * se
    return float(mean), (float(lo), float(hi))


In [ ]:
def scores_from_examples(examples: dict, score_key: str) -> dict:
    out = {}
    for ex_id, obj in examples.items():
        if score_key in obj and obj[score_key] is not None:
            out[ex_id] = float(obj[score_key])
    return out


def ua_vs_ru_cell_stats(
    ua_examples: dict,
    ru_examples: dict,
    ua_key="score_ua_mt_vs_ua_human",
    ru_key="score_ru_mt_vs_ru_human",
    do_wilcoxon=True,
):
    ua_scores = scores_from_examples(ua_examples, ua_key)
    ru_scores = scores_from_examples(ru_examples, ru_key)

    common = sorted(set(ua_scores) & set(ru_scores))
    if len(common) == 0:
        raise ValueError("No overlapping ids between UA and RU examples dicts.")

    # x = UA distances, y = RU distances
    x = np.array([ua_scores[i] for i in common], dtype=float)
    y = np.array([ru_scores[i] for i in common], dtype=float)

    diff = y - x
    mean_diff, (ci_lo, ci_hi) = paired_mean_ci(diff)

    res = {
        "n_common": int(len(common)),
        "mean_UA": float(x.mean()),
        "mean_RU": float(y.mean()),
        "mean_diff_RU_minus_UA": mean_diff,
        "ci95_diff_RU_minus_UA": (ci_lo, ci_hi),
        "median_diff_RU_minus_UA": float(np.median(diff)),
    }

    # Paired t-test consistent with RU-UA
    tt = ttest_rel(y, x)
    res["paired_ttest"] = {"t": float(tt.statistic), "p": float(tt.pvalue)}

    if do_wilcoxon:
        try:
            w = wilcoxon(y, x, zero_method="wilcox")
            res["wilcoxon"] = {"W": float(w.statistic), "p": float(w.pvalue)}
        except ValueError as e:
            res["wilcoxon"] = {"error": str(e)}

    return res


In [ ]:
def compute_table_cells(all_models, all_groups, base_results_dir, ua_key, ru_key):
    """
    Returns a list of dicts, one per (model, group) cell:
    includes p (raw) and later will include q (BH-adjusted).
    """
    cells = []

    for model in all_models:
        model_dir = f"{base_results_dir}/{model}"

        for group_name, subdirs in all_groups.items():
            print(f"Processing model={model}, group={group_name}, model_dir={model_dir}, subdirs={subdirs}...")
            ua_files, ru_files = collect_latest_runs_for_subdirs(model_dir, subdirs)
            print("Latest UA files:", ua_files)
            print("Latest RU files:", ru_files)
            ua_examples = load_all_examples_with_prefix(ua_files)
            ru_examples = load_all_examples_with_prefix(ru_files)
            print(f"Processing model={model}, group={group_name} with {len(ua_examples)} UA and {len(ru_examples)} RU examples.")
            #print 5 examples of each
            print("UA examples:")
            for i, ex in enumerate(ua_examples.values()):
                if i < 5:
                    print(f"  {ex}")
            print("RU examples:")
            for i, ex in enumerate(ru_examples.values()):
                if i < 5:
                    print(f"  {ex}")

            stats = ua_vs_ru_cell_stats(
                ua_examples, ru_examples,
                ua_key=ua_key, ru_key=ru_key,
                do_wilcoxon=True
            )

            cell = {
                "model": model,
                "group": group_name,
                **stats,
            }
            cells.append(cell)

    # BH across ALL cells in this table (8*5 = 40)
    pvals = [c["paired_ttest"]["p"] for c in cells]
    qvals = bh_fdr(pvals)
    for c, q in zip(cells, qvals):
        c["paired_ttest"]["q_bh"] = float(q)
        c["sig_q_0.05"] = (q < 0.05)

    return cells


def print_cells(cells):
    for c in sorted(cells, key=lambda z: (z["group"], z["model"])):
        star = "*" if c["sig_q_0.05"] else ""
        lo, hi = c["ci95_diff_RU_minus_UA"]
        print(
            f"{c['group']:>16} | {c['model']:<22} | "
            f"RU/UA={c['mean_RU']:.3f}/{c['mean_UA']:.3f} | "
            f"delta(RU-UA)={c['mean_diff_RU_minus_UA']:+.4f} "
            f"[{lo:+.4f},{hi:+.4f}] | "
            f"p={c['paired_ttest']['p']:.3g} q={c['paired_ttest']['q_bh']:.3g}{star}"
        )
